# 01. Build the crop panel

**Purpose:** turn the raw FAOSTAT download into one clean table of area, production and
yield by country, crop and year.

**Input:** FAOSTAT Production Crops and Livestock, Africa. Downloads on first run.

**Output:** `data/processed/crop_panel.csv`

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np

from src import config as C
from src import io_utils as io

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

## 1. Load

The file downloads itself the first time and is cached after that.

I read it as utf-8. latin-1 also reads it without any error, but it quietly turns
`Côte d'Ivoire` into `CÃ´te d'Ivoire`, so `load_faostat` checks for that and stops.

In [2]:
raw = io.load_faostat("production")
print("shape:", raw.shape)
raw[["Area", "Item", "Element", "Unit", "Y1990", "Y2023"]].head()

saved    Production_Crops_Livestock_E_Africa.csv  (17.6 MB)


shape: (14750, 201)


,Area,Item,Element,Unit,Y1990,Y2023
0,Algeria,"Almonds, in shell",Area harvested,ha,25010.0,38540.00
1,Algeria,"Almonds, in shell",Yield,kg/ha,469.1,1809.40
2,Algeria,"Almonds, in shell",Production,t,11733.0,69734.97
3,Algeria,Apples,Area harvested,ha,10740.0,32852.00
4,Algeria,Apples,Yield,kg/ha,3625.8,17704.20


The file is wide. Nine columns describe the row, then three columns per year: the
value, a quality flag, and a note.

In [3]:
print("describing columns:", list(raw.columns[:9]))
print("then per year:      ", list(raw.columns[9:12]))
print()
print("years:       ", len(io.year_columns(raw)))
print("countries:   ", raw["Area"].nunique())
print("items:       ", raw["Item"].nunique())
print("measurements:", list(raw["Element"].unique()))

describing columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Unit']
then per year:       ['Y1961', 'Y1961F', 'Y1961N']

years:        64
countries:    57
items:        273
measurements: ['Area harvested', 'Yield', 'Production', 'Stocks', 'Producing Animals/Slaughtered', 'Yield/Carcass Weight', 'Laying', 'Milk Animals']


Livestock is in here too, so I filter on measurement as well as crop.

### Data quality

FAOSTAT marks every value: `A` official, `E` estimated, `I` imputed, `X` from another
organisation.

Nigeria has no reliable agricultural census, so a lot of its numbers are estimates. The
long-run trends are still worth looking at. Single years are not.

In [4]:
ng = raw[raw["Area"] == "Nigeria"]
pd.DataFrame({"1990": ng["Y1990F"].value_counts(),
              "2023": ng["Y2023F"].value_counts()}).fillna(0).astype(int)

,1990,2023
A,123,3
E,117,140
I,20,104
M,0,5
X,9,31


## 2. Pick the crops and countries

Eight staples, fifteen countries. Both lists are in `src/config.py`.

Egypt and South Africa are not like the others. Egypt irrigates nearly everything and
South Africa farms commercially, while Nigeria mostly farms small rain-fed plots. They
are in to show the best that is possible, not as a target.

In [5]:
for group, members in C.PEER_GROUPS.items():
    print(f"{group:<26} {', '.join(members)}")

Neighbours                 Ghana, Benin, Cameroon, Niger
West Africa                Mali, Burkina Faso, Senegal, Côte d'Ivoire
East Africa                Ethiopia, Kenya, Uganda, United Republic of Tanzania
Irrigated or commercial    Egypt, South Africa


A name spelt wrong does not raise an error. It just returns nothing, and that crop
disappears from the whole project. So I check the names first.

In [6]:
missing = ([c for c in C.CROPS if c not in set(raw["Item"])]
           + [p for p in C.PEERS if p not in set(raw["Area"])])

if missing:
    raise ValueError(f"not found in FAOSTAT, check src/config.py: {missing}")

print(f"Verified {len(C.CROPS)} crops and {len(C.PEERS)} countries.")

Verified 8 crops and 15 countries.


In [7]:
df = raw[
    raw["Item"].isin(C.CROPS)
    & raw["Area"].isin(C.PEERS)
    & raw["Element"].isin(["Area harvested", "Production", "Yield"])
]

most = len(C.CROPS) * len(C.PEERS) * 3
print(f"rows: {len(df)}   most it could be: {most}")

rows: 330   most it could be: 360


Fewer than the maximum because not every country grows every crop.

## 3. Reshape

Year is a column heading at the moment, so I cannot filter or group on it. Melting moves
it into a column. The flag and note columns are left out.

In [8]:
long = io.faostat_to_long(df, ["Area", "Item", "Element", "Unit"])
print("rows:", len(long))
print(long.dtypes.to_string())

rows: 19776
Area        object
Item        object
Element     object
Unit        object
year         int64
value      float64


Blank values are dropped rather than filled with zero. A country that reported
nothing did not harvest nothing, the number just does not exist.

In [9]:
long["crop"] = long["Item"].map(C.CROPS)

if long["crop"].isna().any():
    raise ValueError("a crop did not map to a short name")

long[["Item", "crop"]].drop_duplicates().sort_values("crop").reset_index(drop=True)

,Item,crop
0,"Cassava, fresh",Cassava
1,"Cow peas, dry",Cowpea
2,"Groundnuts, excluding shelled",Groundnut
3,Maize (corn),Maize
4,Millet,Millet
5,Rice,Rice
6,Sorghum,Sorghum
7,Yams,Yam


## 4. Pivot

Area, production and yield come as separate rows. I need them as columns so I can
calculate across them.

`pivot_table` rather than `pivot` because it does not fall over on duplicate keys.
`aggfunc="first"` keeps the real value instead of averaging.

In [10]:
panel = (
    long.pivot_table(index=["Area", "crop", "year"], columns="Element",
                     values="value", aggfunc="first")
    .reset_index()
    .rename(columns={"Area": "country",
                     "Area harvested": "area_ha",
                     "Production": "production_t",
                     "Yield": "yield_published_kg_ha"})
)
panel.columns.name = None

print("shape:", panel.shape)
panel.head(3)

shape: (6592, 6)


,country,crop,year,area_ha,production_t,yield_published_kg_ha
0,Benin,Cassava,1961,74000.0,280000.0,3783.8
1,Benin,Cassava,1962,82000.0,320000.0,3902.4
2,Benin,Cassava,1963,84000.0,330000.0,3928.6


I rename `Area` to `country` straight away. FAOSTAT uses it for the country, and it
would otherwise sit next to `area_ha`, which is hectares.

## 5. Check the published yield

Yield is just production divided by area, so I can work it out myself and see whether it
matches theirs.

Mine also comes out in tonnes per hectare, so there is no kg conversion to get wrong.

In [11]:
panel["yield_t_ha"] = panel["production_t"] / panel["area_ha"]

published = panel["yield_published_kg_ha"] / 1000
gap = ((panel["yield_t_ha"] - published).abs() / published * 100)

print(f"biggest difference: {gap.max():.3f}%")
print(f"rows over 1%:       {(gap > 1).sum()}")

if gap.max() > 1:
    raise ValueError("my yield and the published yield disagree")

biggest difference: 0.359%
rows over 1%:       0


They agree everywhere, so I use my own column from here.

## 6. Drop what cannot be real

Two filters. The first stops a divide by zero. The second removes yields no real harvest
of these crops could produce.

0.05 t/ha is 50 kg off a hectare, which is not a harvest. 100 t/ha is above what cassava
or yam can do. These come from the crops, not from a statistical rule.

I look at anything before deleting it.

In [12]:
before = len(panel)
panel = panel[(panel["area_ha"] > 0) & (panel["production_t"] > 0)]
print("zero area or production removed:", before - len(panel))

lo, hi = C.YIELD_BOUNDS
panel[~panel["yield_t_ha"].between(lo, hi)][
    ["country", "crop", "year", "area_ha", "production_t", "yield_t_ha"]]

zero area or production removed: 0


,country,crop,year,area_ha,production_t,yield_t_ha
3925,Niger,Cowpea,1993,3364470.0,163920.0,0.048721


In [13]:
before = len(panel)
panel = panel[panel["yield_t_ha"].between(*C.YIELD_BOUNDS)]
print(f"removed {before - len(panel)} row(s)")

removed 1 row(s)


One row. Niger, cowpea, 1993: 3.4 million hectares producing 49 kg per hectare.
Cowpea in that region runs 300 to 600 kg per hectare, so it is either a disaster or a
reporting error. Either way it should not sit in an average.

## 7. Save

Sorted because notebook 03 takes the first and last three rows of each crop, which only
works if the rows are in year order.

In [14]:
panel = panel[["country", "crop", "year", "area_ha", "production_t", "yield_t_ha"]]
panel = panel.sort_values(["country", "crop", "year"]).reset_index(drop=True)

panel.to_csv(C.PROCESSED / "crop_panel.csv", index=False)

print("rows:     ", len(panel))
print("countries:", panel["country"].nunique())
print("crops:    ", panel["crop"].nunique())
print("years:    ", panel["year"].min(), "to", panel["year"].max())
panel.head()

rows:      6591
countries: 15
crops:     8
years:     1961 to 2024


,country,crop,year,area_ha,production_t,yield_t_ha
0,Benin,Cassava,1961,74000.0,280000.0,3.783784
1,Benin,Cassava,1962,82000.0,320000.0,3.902439
2,Benin,Cassava,1963,84000.0,330000.0,3.928571
3,Benin,Cassava,1964,90000.0,350000.0,3.888889
4,Benin,Cassava,1965,98000.0,400000.0,4.081633


---

**Next:** `02_context.ipynb` adds population and fertilizer.